# Season outputs loader
Helper cells to flexibly load season run outputs stored under `data/season_outputs/<run_id>`.
- Lists available runs
- Loads trip log, day summary, season person snapshots, SP day summary
- Discovers all `day_*_model_ts.parquet` files into a dict keyed by day index

Update `RUN_ID` below to point at the run you want to analyze.

In [ ]:
import os
from pathlib import Path
import subprocess

# Get the top-level directory of the current git repo
PROJECT_ROOT = Path(
    subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], text=True
    ).strip()
)

os.chdir(PROJECT_ROOT)


# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

%pwd


In [ ]:
from pathlib import Path
import re
import pandas as pd
import seaborn as sns 
import numpy as np

BASE_DIR = Path('data/season_outputs')
 # e.g., 'my_season_id'

def list_runs(base_dir: Path = BASE_DIR):
    if not base_dir.exists():
        return []
    return sorted([p.name for p in base_dir.iterdir() if p.is_dir()])

def _read_parquet_optional(path: Path):
    if not path.exists():
        return None
    return pd.read_parquet(path)

def load_model_ts(run_dir: Path):
    model_ts = {}
    if not run_dir.exists():
        return model_ts
    pattern = re.compile(r'^day_(\d+)_model_ts\.parquet$')
    for p in sorted(run_dir.glob('day_*_model_ts.parquet')):
        m = pattern.match(p.name)
        if not m:
            continue
        day_idx = int(m.group(1))
        model_ts[day_idx] = pd.read_parquet(p)
    return model_ts

def load_run(run_id: str, base_dir: Path = BASE_DIR):
    run_dir = base_dir / run_id
    data = {
        'run_dir': run_dir,
        'trip_log': _read_parquet_optional(run_dir / 'trip_log.parquet'),
        'day_summary': _read_parquet_optional(run_dir / 'day_summary.parquet'),
        'season_person_log': _read_parquet_optional(run_dir / 'season_person_log.parquet'),
        'sp_day_summary': _read_parquet_optional(run_dir / 'sp_day_summary.parquet'),
        'model_ts': load_model_ts(run_dir),
    }
    return data

available_runs = list_runs()
available_runs


## Load a run
Set `RUN_ID` to one of the `available_runs` above.

In [ ]:
RUN_ID = 'volume_toll_6' 

run_data = load_run(RUN_ID)
run_data_keys = {k: (list(v.keys()) if k == 'model_ts' else (None if v is None else getattr(v, 'shape', None))) for k, v in run_data.items()}
run_data_keys


## Quick peeks
Uncomment and run the snippets you need once a run is loaded.

In [ ]:
trip_log = run_data['trip_log']
day_summary = run_data['day_summary']

model_ts = run_data['model_ts']


model_ts[0].head()
# season_person_log = run_data['season_person_log']
# sp_day_summary = run_data['sp_day_summary']
# # display(day_summary.head()) if day_summary is not None else None
# display(trip_log.head()) if trip_log is not None else None
# display(sp_day_summary.head()) if sp_day_summary is not None else None
# list(model_ts.keys())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_realized_cost_means_with_total(trip_log):
    # mean by day and mode
    summary_mode = (
        trip_log
        .groupby(["day_index", "mode"], as_index=False)["realized_cost"]
        .mean()
    )

    # overall mean by day (all modes combined)
    summary_total = (
        trip_log
        .groupby("day_index", as_index=False)["realized_cost"]
        .mean()
        .rename(columns={"realized_cost": "mean_cost_total"})
    )

    fig, ax = plt.subplots(figsize=(10, 6))

    color_map = {"car": "red", "bus": "blue"}

    # mode-specific lines
    for mode in summary_mode["mode"].unique():
        sub = summary_mode[summary_mode["mode"] == mode].sort_values("day_index")
        ax.plot(
            sub["day_index"],
            sub["realized_cost"],
            label=f"{mode} mean",
            color=color_map.get(mode, None),
        )

    # total line (all persons, all modes)
    sub_tot = summary_total.sort_values("day_index")
    ax.plot(
        sub_tot["day_index"],
        sub_tot["mean_cost_total"],
        label="total mean",
        color="black",
        linestyle="--",
    )

    ax.set_xlabel("Day")
    ax.set_ylabel("Realized cost")
    ax.set_title("Average realized cost by mode and overall")
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


In [ ]:
plot_realized_cost_means_with_total(trip_log)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(14, 6))

# Boxplots: 2 per day (car vs bus)
sns.boxplot(
    data=trip_log,
    x="day_index",
    y="realized_cost",
    hue="mode",
    palette={"car": "red", "bus": "blue"},
    ax=ax,
)

# Vertical separators between days
xticks = ax.get_xticks()
for i in range(len(xticks) - 1):
    mid = (xticks[i] + xticks[i + 1]) / 2
    ax.axvline(mid, color="black", linewidth=1, linestyle="-", alpha=0.6)

ax.set_xlabel("Day")
ax.set_ylabel("Realized cost")
ax.set_title("Realized cost by mode and day")
ax.legend(title="Mode")

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

import pandas as pd
import plotly.graph_objects as go

def plot_model_ts_interactive(model_ts):
    # Flatten dict of day -> df into one DataFrame
    dfs = []
    for day, df in model_ts.items():
        tmp = df.copy()
        tmp["day_index"] = day
        dfs.append(tmp)
    data = pd.concat(dfs, ignore_index=True)

    # Time in minutes
    data["minute"] = data["Step"] / 60.0

    metrics = [
        "volume",
        "current_toll_car",
        "at_bus_stop",
        "avg_posted_sl_delta",
        "bus_mode_share_recent",
    ]
    start_metric = "volume"
    days = sorted(data["day_index"].unique())
    n_days = len(days)

    # Simple red -> blue gradient: day 0 = red, last day = blue
    colors = []
    for i in range(n_days):
        t = i / (n_days - 1) if n_days > 1 else 0
        r = int(255 * (1 - t))   # 255 -> 0
        g = 0
        b = int(255 * t)         # 0 -> 255
        colors.append(f"rgb({r},{g},{b})")

    fig = go.Figure()

    # One trace per day (default metric = volume)
    for i, day in enumerate(days):
        sub = data[data["day_index"] == day]
        fig.add_trace(
            go.Scatter(
                x=sub["minute"],
                y=sub[start_metric],
                mode="lines",
                name=f"Day {day}",
                line=dict(color=colors[i]),
            )
        )

    # Build dropdown to switch metric (y-values)
    buttons = []
    for metric in metrics:
        new_ys = []
        for day in days:
            sub = data[data["day_index"] == day]
            new_ys.append(sub[metric])

        buttons.append(
            dict(
                label=metric,
                method="update",
                args=[
                    {"y": new_ys},                  # update all traces' y-data
                    {"yaxis": {"title": metric}},   # update y-axis label
                ],
            )
        )

    fig.update_layout(
        title="Model time series by day",
        xaxis_title="Minute",
        yaxis_title=start_metric,
        legend_title_text="Day (click to show/hide)",
        updatemenus=[
            dict(
                buttons=buttons,
                direction="down",
                x=1.05,
                xanchor="left",
                y=1,
                yanchor="top",
                showactive=True,
            )
        ],
    )

    fig.show()



In [ ]:
plot_model_ts_interactive(model_ts)